In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

sys.path.insert(0, str(next((p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / 'utils').exists()), Path().resolve())))
from utils.nlp import compute_tfidf_cosine_similarity
from utils.imports import import_excel


In [ ]:
raw_drug_names_df = import_excel('data/drug_names.xlsx')
drug_cat_df = import_excel('data/drug_catalog.xlsx')

In [3]:
drug_cat_df['atc_drug_name'] = drug_cat_df['Drug Name'].astype(str).str.lower()
raw_drug_names_df['text_drug_name'] = raw_drug_names_df['drug_name'].astype(str).str.lower()

In [4]:
res_df = compute_tfidf_cosine_similarity(raw_drug_names_df['text_drug_name'].tolist(), drug_cat_df['atc_drug_name'].tolist())

In [5]:
res_df = res_df.merge(drug_cat_df.drop(columns={'Drug Name'}), how='left', on='atc_drug_name')

In [6]:
final_res_df = res_df[['text_drug_name', 'match_rate']].groupby('text_drug_name').max().reset_index().merge(
    res_df, how='inner', on=['text_drug_name', 'match_rate'])

In [7]:
px.histogram(final_res_df['match_rate'], nbins=10)

In [9]:
final_res_df.loc[final_res_df['match_rate'] >= 0.8, 'correct_match'] = True
final_res_df.loc[final_res_df['match_rate'] < 0.8, 'correct_match'] = False

In [12]:
final_res_df.correct_match.value_counts()

correct_match
False    229
True      93
Name: count, dtype: int64